# 02 · Benchmark B — infinite well and the periodic-versus-Dirichlet comparison

**Scientific question.** Does the choice of spectral transform change the physics represented, and by how much?

**Scope.** Zero-potential hard-wall well as a control, plus the direct comparison of periodic and Dirichlet propagation of the same problem against an independent reference.

**Inputs.** `configs/{PROFILE}.yaml` (loaded below). No other notebook needs to have
been run first: this notebook imports everything it needs from
`src/boundary_aware_dynamics` and holds no state from any other notebook.

**Expected outputs.** Density snapshots under both topologies, error and wall-residual curves, and the numbers behind the project's central claim.

**Approximate runtime.** about 20 seconds on the `smoke` profile.

**Method.** Both propagators are given the same initial state, box, resolution, interval and step count, and differ only in which transform sits inside the split step. Both are compared against a finite-difference hard-wall reference sharing neither basis nor method with either.

**Assumptions.** The interval is long enough for the packet to reach a wall — otherwise the two topologies would agree trivially.

**References.** See `references/references.bib` and `docs/SCIENTIFIC_METHOD.md`.

**What this notebook does _not_ establish.** This is **not** a Trotter benchmark. With zero interior potential the Dirichlet propagator is exact, so a step-count sweep here measures nothing about the time integrator. Benchmark C is the Trotter benchmark.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt

from boundary_aware_dynamics.config import load_config
from boundary_aware_dynamics import plotting

PROFILE = os.environ.get("BAD_PROFILE", "smoke")   # "paper" for manuscript numbers
                                                   # (scripts/execute_notebooks.py sets this)
config = load_config(ROOT / "configs" / f"{PROFILE}.yaml")
plotting.apply_style("preview")
print(f"profile={config.profile}  config_hash={config.config_hash}")

## Why this benchmark is a control

The DST-II diagonalises the Dirichlet Laplacian, and with $V=0$ there is no non-commuting splitting left. The propagator is therefore *exact*, at any step count. Demonstrated rather than asserted:

In [ ]:
from boundary_aware_dynamics.workflows import run_benchmark, trotter_convergence_study, run_boundary_comparison

study = trotter_convergence_study(config, "infinite_well")
for r, err in zip(study.values, study.l2_state_error):
    print(f"  r={r:5d}  L2 state error {err:.3e}")
print(f"\nfitted slope: {study.fit.get('slope')}  ({study.fit.get('note','')})")
print("Every value is at the round-off floor: there is no Trotter error to converge.")

### The near-perfect fidelity here is structural

The DST-II rows *are* the analytical sine eigenmodes sampled on the midpoint grid, and the propagator uses the same eigenvalues as the sine-series reference. Agreement is therefore built in and is **not** independent evidence of continuum accuracy.

In [ ]:
from boundary_aware_dynamics.transforms import analytical_dst2_matrix, dst2_matrix
from boundary_aware_dynamics.grids import dirichlet_midpoint_grid
from boundary_aware_dynamics.references import sine_basis

N, L = 32, config.benchmark("infinite_well").domain.length
g = dirichlet_midpoint_grid(L, N)
rows = dst2_matrix(N)
modes = sine_basis(g.positions, L, N) * np.sqrt(g.spacing)
modes[-1] /= np.sqrt(2.0)          # Nyquist row normalisation
print(f"|| DST-II rows - sampled sine eigenmodes || = {np.abs(rows - modes).max():.2e}")
print("=> the propagator eigenbasis IS the reference eigenbasis; the match is circular.")

## The direct comparison

Same state, same box, same grid, same times, same step count. Only the transform differs. Both are judged against a finite-difference reference that uses neither the sine basis nor the split-operator method.

In [ ]:
comparison = run_boundary_comparison(config, "infinite_well")
d_inf = np.array([e.infidelity for e in comparison.dirichlet_errors])
p_inf = np.array([e.infidelity for e in comparison.periodic_errors])
d_wall = np.array([b["wall_residual"] for b in comparison.dirichlet_boundary])
p_wall = np.array([b["wall_residual"] for b in comparison.periodic_boundary])
cross = comparison.cross_fidelity

print(f"reference: {comparison.reference_method}\n")
print(f"{'t':>6} {'Dirichlet infid':>16} {'periodic infid':>16} {'cross fidelity':>15}")
for i in np.linspace(0, len(comparison.times) - 1, 6).astype(int):
    print(f"{comparison.times[i]:6.2f} {d_inf[i]:16.3e} {p_inf[i]:16.3e} {cross[i]:15.4f}")
print(f"\nfinal: Dirichlet {d_inf[-1]:.2e} vs periodic {p_inf[-1]:.2e} "
      f"-> ratio {p_inf[-1]/max(d_inf[-1],1e-18):.3g}")
print(f"minimum cross-fidelity between the two propagations: {cross.min():.4f}")

In [ ]:
fig = plotting.plot_boundary_comparison(comparison.times, d_inf, p_inf, cross, d_wall, p_wall)
plt.show()

In [ ]:
fig = plotting.plot_density_snapshots(
    comparison.grid.positions, comparison.times,
    {"reference": comparison.reference_states,
     "dirichlet": comparison.dirichlet_states,
     "periodic": comparison.periodic_states},
    np.unique(np.linspace(0, len(comparison.times) - 1, 4).astype(int)))
plt.show()

## Wrap-around: what the periodic model does wrong

A ring has no walls, so amplitude leaving one end reappears at the other. Near-wall probability is *not* an error — the exact solution has probability there too — but wrap-around and a large wall residual are.

In [ ]:
d_wrap = np.array([b["wrap_around_probability"] for b in comparison.dirichlet_boundary])
p_wrap = np.array([b["wrap_around_probability"] for b in comparison.periodic_boundary])
print(f"max wall residual  : Dirichlet {d_wall.max():.3f}   periodic {p_wall.max():.3f}")
print(f"max edge probability: Dirichlet {d_wrap.max():.3f}   periodic {p_wrap.max():.3f}")

## Does raising the resolution rescue the periodic model?

No. Increasing $N$ refines a representation of the *wrong topology*; it cannot introduce walls that the model does not have.

In [ ]:
from boundary_aware_dynamics.grids import periodic_grid, Grid
from boundary_aware_dynamics.propagators import split_operator_evolution
from boundary_aware_dynamics.references import finite_difference_reference
from boundary_aware_dynamics.diagnostics import fidelity
from boundary_aware_dynamics.workflows import build_initial_state, build_state_callable

bench = config.benchmark("infinite_well")
t_max, steps = bench.time_grid.t_max, bench.time_grid.n_steps
print(f"{'N':>5} {'Dirichlet infid':>17} {'periodic infid':>16}")
for n in (16, 32, 64, 128):
    gd = dirichlet_midpoint_grid(L, n)
    ring = Grid(gd.positions, gd.spacing, "periodic", (0.0, L))
    psi0 = build_initial_state(bench, gd)
    times = np.linspace(0.0, t_max, steps + 1)
    ref = finite_difference_reference(build_state_callable(bench), gd, times,
                                      config.physics.mass, config.physics.hbar, None, 8)
    a = split_operator_evolution(psi0, gd, None, t_max, steps, config.physics.mass, config.physics.hbar)
    b = split_operator_evolution(psi0, ring, None, t_max, steps, config.physics.mass, config.physics.hbar)
    print(f"{n:5d} {1-fidelity(ref.final_state, a.final_state, gd.spacing):17.3e} "
          f"{1-fidelity(ref.final_state, b.final_state, gd.spacing):16.3e}")

## Summary

**Main findings.** The two transforms give materially different dynamics for the identical problem once the packet reaches a wall: the Dirichlet propagation tracks the independent hard-wall reference to ~1e-5 infidelity while the periodic propagation reaches order-unity error, and the two propagations become nearly orthogonal. Refining the grid does not help the periodic model, because the defect is topological rather than a resolution deficit.

**Validation checks performed.** Step-count independence of the free well; the circularity of its fidelity, shown explicitly; comparison of both propagators against an independent finite-difference reference; wall residual, wrap-around probability and a resolution sweep.

**Limitations.** This benchmark has zero interior potential, so it says nothing about Trotter error. The claim it supports is about representation, not about integrator accuracy.

**Generated files.** None directly; notebook 05 exports these figures.

**Relationship to the manuscript.** This is the notebook behind the paper's central quantitative claim.

**Next.** `03_tilted_infinite_well.ipynb` adds an interior potential so that a genuine Trotter benchmark exists under hard walls.